# Python for data engineering

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-09-data-engineering.ipynb)

The earlier chapters are general Python. This one is about the habits that keep a data pipeline correct when it runs again, when part of its input is wrong, and when nobody is watching. Each habit is a property you can test:

- **Idempotent:** running a job twice leaves the same result as running it once.
- **Restartable:** a crash leaves no half-finished state, and the next run picks up where the last good run ended.
- **Validated:** every bad record is counted and kept aside. None is silently dropped, and none breaks the load.
- **Observable:** every run logs what it read, loaded and rejected.

Each idea is built here with the standard library only, so every cell runs on Colab and locally. For columnar files see [Parquet](../../parquet/parquet.ipynb), for scheduling see [Airflow](../../airflow/airflow.ipynb), and for quality rules see [Data quality](../../concepts/data-quality.ipynb).

## Configuration from the environment

Pipelines run in many places, so settings come from environment variables, each with a safe default. Read them once, at start-up, and validate them there. A typo such as `BATCH_SIZE=0` should stop the job before it reads a single row. Every environment value is a string, so convert each one and check the result.

### Where you'll use it

- **Twelve-factor deployments:** the same image runs in development, staging and production, with different environment variables.
- **Kubernetes and Airflow:** settings arrive as environment variables or secrets, not as values written into the code.

In [ ]:
MIN_BATCH_SIZE = 1
MIN_RETRIES = 1
DEFAULT_BATCH_SIZE = 500
DEFAULT_RETRIES = 3

def load_config(env):
    config = {
        "batch_size": int(env.get("BATCH_SIZE", str(DEFAULT_BATCH_SIZE))),
        "max_retries": int(env.get("MAX_RETRIES", str(DEFAULT_RETRIES))),
        "landing_dir": env.get("LANDING_DIR", "/data/landing"),
    }
    if config["batch_size"] < MIN_BATCH_SIZE:
        raise ValueError("BATCH_SIZE must be at least 1")
    if config["max_retries"] < MIN_RETRIES:
        raise ValueError("MAX_RETRIES must be at least 1")
    return config

assert load_config({})["batch_size"] == DEFAULT_BATCH_SIZE
assert load_config({"BATCH_SIZE": "50"})["batch_size"] == 50

try:
    load_config({"BATCH_SIZE": "0"})
except ValueError as error:
    assert "BATCH_SIZE" in str(error)
else:
    raise AssertionError("expected ValueError")

## Validate records and quarantine the rejects

A record is either **loaded**, **skipped** (already processed, covered in the watermark section), or **quarantined** with the reason it failed. Quarantine keeps the bad data for investigation, and it keeps the counts honest: every record read must be accounted for, so `read = loaded + skipped + quarantined` always holds.

Money is parsed from text with `Decimal` and stored as integer cents, so no float error reaches the table. A parser should raise a clear `ValueError` for bad input, and the pipeline should catch that error and quarantine the record.

### Where you'll use it

- **Data contracts:** a validation step rejects rows that break the agreed schema, and keeps them for the producer to fix.
- **Audit trails:** a quarantine table with a reason per row means every rejected record can be traced back to its source.

In [ ]:
import json
from decimal import Decimal, InvalidOperation

def parse_amount_cents(text):
    try:
        amount = Decimal(str(text))
    except InvalidOperation as error:        # Decimal raises InvalidOperation, not ValueError
        raise ValueError(f"not a number: {text!r}") from error
    return int((amount * 100).to_integral_value())

def parse_record(raw):
    order_id = int(raw["order_id"])
    amount_cents = parse_amount_cents(raw["amount"])
    updated_at = str(raw["updated_at"])
    if amount_cents < 0:
        raise ValueError("negative amount")
    if not updated_at:
        raise ValueError("missing updated_at")
    return {"order_id": order_id, "amount_cents": amount_cents, "updated_at": updated_at}

good = parse_record(json.loads('{"order_id": "7", "amount": "12.50", "updated_at": "2024-01-15T10:00:00"}'))
assert good == {"order_id": 7, "amount_cents": 1250, "updated_at": "2024-01-15T10:00:00"}

for bad_line in ['{"order_id": "x", "amount": "1", "updated_at": "t"}',
                 '{"order_id": 1, "amount": "oops", "updated_at": "t"}',
                 '{"order_id": 1, "amount": "-3", "updated_at": "t"}',
                 '{"order_id": 1, "amount": "2"}']:
    try:
        parse_record(json.loads(bad_line))
    except (KeyError, ValueError):
        pass
    else:
        raise AssertionError(f"should have been rejected: {bad_line}")

## Idempotent loads with an upsert

An **upsert** inserts a row, or updates the existing row when its key is already present. Loading the same batch twice then leaves the table unchanged, which is what makes a retry safe.

The `WHERE` guard on the update stops an **older** version of a record from overwriting a newer one. That matters when late data arrives out of order. This example uses SQLite, which ships with Python. The same `INSERT ... ON CONFLICT` pattern exists in PostgreSQL.

### Where you'll use it

- **Retrying failed loads:** an Airflow task that retries after a timeout must not double-count orders.
- **Change data capture targets:** a sink that receives inserts and updates from a CDC stream applies them with the same upsert.

In [ ]:
import sqlite3

def make_db():
    db = sqlite3.connect(":memory:")
    db.execute("""
        CREATE TABLE orders (
            order_id     INTEGER PRIMARY KEY,
            amount_cents INTEGER NOT NULL,
            updated_at   TEXT NOT NULL
        )
    """)
    return db

def upsert(db, records):
    db.executemany(
        """
        INSERT INTO orders (order_id, amount_cents, updated_at) VALUES (:order_id, :amount_cents, :updated_at)
        ON CONFLICT(order_id) DO UPDATE SET
            amount_cents = excluded.amount_cents,
            updated_at   = excluded.updated_at
        WHERE excluded.updated_at >= orders.updated_at
        """,
        records,
    )
    db.commit()

db = make_db()
batch = [
    {"order_id": 1, "amount_cents": 500, "updated_at": "2024-01-10"},
    {"order_id": 2, "amount_cents": 900, "updated_at": "2024-01-11"},
]
upsert(db, batch)
upsert(db, batch)                                          # running it again changes nothing
assert db.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 2

upsert(db, [{"order_id": 1, "amount_cents": 700, "updated_at": "2024-01-12"}])   # newer: applied
upsert(db, [{"order_id": 1, "amount_cents": 1, "updated_at": "2024-01-01"}])     # older: ignored
assert db.execute("SELECT amount_cents FROM orders WHERE order_id = 1").fetchone()[0] == 700

## Incremental processing with a watermark

Reprocessing everything every night is expensive. A **watermark** is the newest `updated_at` already loaded. Each run reads only the records newer than the watermark.

The order of the steps matters. Load the records first, and **move the watermark only after the load commits**. If the run crashes, the watermark stays where it was, and the next run retries the same records. The upsert makes that retry safe.

### Where you'll use it

- **Nightly loads:** load only the rows changed since the last successful run, instead of the whole table.
- **Streaming equivalents:** Spark Structured Streaming and Flink track the same idea as a watermark for late events.

In [ ]:
def run_pipeline(raw_lines, db, state, run_id):
    watermark = state.get("watermark", "")
    loaded, skipped, quarantined = [], 0, []

    for line in raw_lines:
        try:
            record = parse_record(json.loads(line))
        except (json.JSONDecodeError, KeyError, ValueError) as error:
            quarantined.append({"line": line, "reason": str(error)})
            continue
        if record["updated_at"] <= watermark:
            skipped += 1
            continue
        loaded.append(record)

    if loaded:
        upsert(db, loaded)
        state["watermark"] = max(r["updated_at"] for r in loaded)   # only after the commit

    summary = {
        "run_id": run_id,
        "read": len(raw_lines),
        "loaded": len(loaded),
        "skipped": skipped,
        "quarantined": len(quarantined),
    }
    assert summary["read"] == summary["loaded"] + summary["skipped"] + summary["quarantined"]
    return summary, quarantined

lines = [
    '{"order_id": 1, "amount": "5.00", "updated_at": "2024-01-10"}',
    '{"order_id": 2, "amount": "oops", "updated_at": "2024-01-11"}',
    '{"order_id": 3, "amount": "8.25", "updated_at": "2024-01-12"}',
]
db = make_db()
state = {}
summary, rejects = run_pipeline(lines, db, state, run_id="run-1")
assert summary == {"run_id": "run-1", "read": 3, "loaded": 2, "skipped": 0, "quarantined": 1}
assert rejects[0]["line"] == lines[1]
assert state["watermark"] == "2024-01-12"

# Rerun the same file: everything is at or below the watermark, so nothing loads twice.
summary, _ = run_pipeline(lines, db, state, run_id="run-2")
assert summary["loaded"] == 0 and summary["skipped"] == 2
assert db.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 2

## Retries with exponential backoff and jitter

Network and database calls fail transiently, so retry them. Three rules keep retries safe:

- Retry **only** the errors that can succeed later. A bad request fails the same way every time.
- Wait longer after each failure. This **exponential backoff** gives a struggling service room to recover.
- Add random **jitter**, so that many workers do not retry at the same moment.

The decorator takes its `sleep` and random source as parameters. A test passes fakes, so the delays are checked exactly and the test does not wait.

### Where you'll use it

- **Calling flaky services:** a database restart or a 503 from a SaaS API is retried with growing waits.
- **Cloud SDKs:** AWS and GCP client libraries use the same pattern internally, and their retry settings use the same ideas.

In [ ]:
import functools
import random
import time

JITTER_LOW = 0.5          # the wait is scaled by a random factor between 0.5 and 1.5
JITTER_SPREAD = 1.0

def retry(attempts=3, base_delay=0.5, retry_on=(ConnectionError,), sleep=time.sleep, random_factor=random.random):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except retry_on:
                    if attempt == attempts:
                        raise
                    exponential = base_delay * 2 ** (attempt - 1)
                    sleep(exponential * (JITTER_LOW + JITTER_SPREAD * random_factor()))
        return wrapper
    return decorator

waits = []
calls = {"n": 0}

@retry(attempts=4, base_delay=1.0, sleep=waits.append, random_factor=lambda: 0.5)   # factor 1.0: no jitter
def flaky_insert():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("database restarting")
    return "inserted"

assert flaky_insert() == "inserted"
assert waits == [1.0, 2.0]          # 1 s, then 2 s: doubling each time

@retry(attempts=2, sleep=waits.append, random_factor=lambda: 0.5)
def bad_request():
    raise ValueError("malformed input")          # not retryable: raised at once

waits_before = len(waits)
try:
    bad_request()
except ValueError:
    assert len(waits) == waits_before            # no wait was added

## Partitioned output and atomic writes

Analytics engines read folders split by a key, such as `dt=2024-01-15/`. This layout lets a query skip whole folders it does not need. Write each partition to a temporary file and move it into place, so a crash never leaves a partial file that a reader could take for a complete one.

### Where you'll use it

- **Data lakes:** Hive-style `dt=` folders are how Spark, Athena, BigQuery external tables and Trino skip data they do not need.
- **Backfills:** rewriting one partition without touching the others.

In [ ]:
import tempfile
from pathlib import Path

def write_partition(root, day, records):
    folder = Path(root) / f"dt={day}"
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / "part-0000.jsonl"
    tmp = folder / "part-0000.jsonl.tmp"
    with tmp.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")
    tmp.replace(target)                          # atomic on one filesystem
    return target

warehouse = Path(tempfile.mkdtemp(prefix="warehouse-"))
write_partition(warehouse, "2024-01-15", [{"order_id": 1}])
write_partition(warehouse, "2024-01-16", [{"order_id": 2}, {"order_id": 3}])

partitions = sorted(p.parent.name for p in warehouse.rglob("*.jsonl"))
assert partitions == ["dt=2024-01-15", "dt=2024-01-16"]
assert not list(warehouse.rglob("*.tmp"))

## Detecting changes with a content hash

To find which rows changed since the last load, hash a **canonical** form of each record. Canonical means the same key order, the same separators and the same encoding every time. Then equal content always gives an equal hash, and a change in any field gives a different one.

### Where you'll use it

- **Snapshot diffs:** compare today's table with yesterday's by hash, to find the changed rows without comparing every column.
- **Deduplication across sources:** hash the business columns to spot the same record arriving from two systems.

In [ ]:
import hashlib

def row_hash(record):
    canonical = json.dumps(record, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

before = {"order_id": 1, "amount_cents": 500}
after = {"amount_cents": 500, "order_id": 1}          # same content, different key order
changed = {"order_id": 1, "amount_cents": 650}

assert row_hash(before) == row_hash(after)
assert row_hash(before) != row_hash(changed)
assert len(row_hash(before)) == 64                    # SHA-256 in hex: 64 characters

## Deduplicating: keep the latest version

Sources send the same record more than once, because of retries, replays and corrections. Deduplicate on the **business key**, and keep the version with the newest timestamp. The comparison uses a strict `>`, so when two versions tie, the first one seen is kept. The result is deterministic.

### Where you'll use it

- **At-least-once sources:** Kafka and most message queues redeliver messages, so every consumer needs this step.
- **Late corrections:** a source resends a corrected order, and the pipeline keeps only the newest version.

In [ ]:
def latest_per_key(records, key, timestamp):
    latest = {}
    for record in records:
        current = latest.get(record[key])
        if current is None or record[timestamp] > current[timestamp]:
            latest[record[key]] = record
    return list(latest.values())

events = [
    {"order_id": 1, "status": "placed", "updated_at": "2024-01-10"},
    {"order_id": 1, "status": "shipped", "updated_at": "2024-01-12"},
    {"order_id": 2, "status": "placed", "updated_at": "2024-01-11"},
    {"order_id": 1, "status": "placed", "updated_at": "2024-01-10"},   # a replay of the first event
]
result = sorted(latest_per_key(events, "order_id", "updated_at"), key=lambda r: r["order_id"])
assert result == [
    {"order_id": 1, "status": "shipped", "updated_at": "2024-01-12"},
    {"order_id": 2, "status": "placed", "updated_at": "2024-01-11"},
]

## Batches with bounded memory and a run summary

Process a large input in batches, and log progress for each batch. Log a summary line at the end of the run, with the run id, so an operator can find the run and compare it with others. A batch is a `list` of at most `size` items, so memory stays bounded however large the input is.

### Where you'll use it

- **Large backfills:** processing a year of history in fixed-size batches, so the worker's memory does not grow with the input.
- **Run reports:** the summary line is what a dashboard or a Slack alert reads after each run.

In [ ]:
import io
import logging

def chunked(iterable, size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

stream = io.StringIO()
logger = logging.getLogger("pipeline.orders")
logger.handlers.clear()
logger.addHandler(logging.StreamHandler(stream))
logger.setLevel(logging.INFO)

def process_in_batches(records, size, run_id):
    total = 0
    for number, batch in enumerate(chunked(records, size), start=1):
        total += len(batch)
        logger.info("run=%s batch=%d rows=%d", run_id, number, len(batch))
    logger.info("run=%s done rows=%d", run_id, total)
    return total

assert process_in_batches(range(5), 2, "run-7") == 5
logged = stream.getvalue().splitlines()
assert logged[0] == "run=run-7 batch=1 rows=2"
assert logged[-1] == "run=run-7 done rows=5"

## Practice questions

1. Show that a record with a missing `amount` is quarantined rather than crashing the run, and that the counts still add up.
2. Explain why the watermark is moved **after** `upsert`, not before it.
3. Write `file_name_for(source, day, sequence)` that returns `"orders_2024-01-15_0003.jsonl"` for `("orders", "2024-01-15", 3)`.

## Answers

### Question 1

`parse_record` raises `KeyError` for a missing field, and `run_pipeline` catches `KeyError` together with the other parse errors. So the record is quarantined, and the three counts still add up.

In [ ]:
db = make_db()
state = {}
summary, rejects = run_pipeline(
    ['{"order_id": 9, "updated_at": "2024-01-20"}',
     '{"order_id": 10, "amount": "1.00", "updated_at": "2024-01-20"}'],
    db, state, run_id="run-3")
assert summary["quarantined"] == 1 and summary["loaded"] == 1
assert summary["read"] == summary["loaded"] + summary["skipped"] + summary["quarantined"]

### Question 2

If the upsert fails after the watermark has moved, the next run skips records that were never written. Moving the watermark last means a failure causes a retry. The idempotent upsert makes that retry safe.

### Question 3

In [ ]:
def file_name_for(source, day, sequence):
    return f"{source}_{day}_{sequence:04d}.jsonl"

assert file_name_for("orders", "2024-01-15", 3) == "orders_2024-01-15_0003.jsonl"

## Interview notes

- **Idempotency is the core property.** Design every load so that a rerun produces the same state. Use upserts keyed by a business key, or overwrite whole partitions.
- **Move state last.** Advance the watermark or the offset only after the data is committed.
- **Quarantine, do not drop.** A rejected row with a reason lets you measure data quality and fix the source.
- **Retry only transient errors,** with backoff and jitter. Retrying a bad request wastes time and can flood a service.
- **Store money as integers,** and make the counts reconcile: `read = loaded + skipped + quarantined`.
- **Make the function testable.** Inject the clock, the random source and the sleep, so tests run fast and check exact behaviour.